# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://github.com/mlcommons/croissant) URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

First, let's inspect all available record sets in the dataset and discover their `@id`s, names, and selected field details. We will use the Croissant API methods and always reference entities using their `@id`.

In [ ]:
# List all main record sets and their field IDs

record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")

for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  @id: {rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields (@id):")
        for field in fields:
            print(f"    - {field['@id']} (name: {field.get('name', 'N/A')})")
    print()

### Example record preview
Let's print a single record from each record set using the `@id`.

*Record set IDs should always be referenced by their `@id`.*

In [ ]:
# Display a first record (if any) from each record set using their @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Sample record from record set @id={rs_id}:")
    try:
        iterator = dataset.records(record_set=rs_id)
        record = next(iterator)
        print(json.dumps(record, indent=2))
    except StopIteration:
        print("  No records found.")
    except Exception as e:
        print(f"  Error: {e}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this notebook, we’ll extract all record sets using their `@id` and store each as a separate DataFrame in a dictionary for downstream analysis.

In [ ]:
# Extract all records from each record set using their @id

record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set @id={rs_id} -> shape: {df.shape}")
    except Exception as e:
        print(f"Could not load record set @id={rs_id}. Error: {e}")

# For demonstration, select the first record set for more detailed exploration
selected_record_set_id = record_set_ids[0] if record_set_ids else None

if selected_record_set_id:
    print(f"\nColumns in record set @id={selected_record_set_id}:\n{dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by attributes.

We will select a numeric field and a group field by inspecting the columns of the main record set (first one in the list).

In [ ]:
# Choose a DataFrame and identify candidate numeric and group fields
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Columns in selected record set (@id): {df.columns.tolist()}")
    
    # Try to automatically select a numeric field and a group field
    numeric_field_id = None
    group_field_id = None
    
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    
    # Try to find a group field (categorical or string)
    for col in df.columns:
        if col != numeric_field_id and (
            df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])
        ):
            group_field_id = col
            break

    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    # EDA only if we have at least one numeric field
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We’ll make a histogram of the selected numeric field and a boxplot by the group field if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot grouped by group_field_id (if any)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the Croissant standard and `mlcroissant` library to load, explore, and visualize the FAIR^2 dataset of clinicopathological and molecular features of second primary colorectal cancer in cancer survivors. We programmatically referenced Croissant entities using their `@id`, demonstrating reproducibility and clarity. This approach can be extended for downstream ML model development, data curation, or further analytics.